# Tutorial 6 — From the structured-jet chain to detection forecasts

This notebook loads the converged **Tutorial 2** chain, generates a full catalogue,
applies the prompt sensitivities, adds the BNS parameters, runs GWFish, and computes
the X-ray afterglows. Set the chain path and simulation duration below, then **Run All**.
The jet fraction comes directly from the selected chain sample.

The default merger-rate file and jet structure match Tutorial 2. For a Tutorial 2.1
chain, point `CHAIN_PATH` to that run and set `STRUCTURE_CSV` to its structure file.
Keep `forecast_pipeline.py` beside the notebook in `Tutorials/`.

In your MAGGPY environment, install the additional tools if needed:

```bash
python -m pip install VegasAfterglow==2.0.4 joblib PyYAML lalsuite matplotlib ipykernel
python -m pip install "GWFish @ git+https://github.com/janosch314/GWFish.git"
```

In [ ]:
from pathlib import Path
from dataclasses import asdict
import sys

import emcee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from maggpy.structured_jet.init import initialize_simulation, load_custom_structure_constants

ROOT = Path.cwd().resolve()
if ROOT.name == 'Tutorials':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'Tutorials'))
import forecast_pipeline as fp

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})

## 1. Settings

`CHAIN_PATH` points to the `emcee.h5` saved by Tutorial 2. `BURN_IN` is the number
of steps to discard. Keep `MRD_PATH` and `STRUCTURE_CSV` consistent with the fitted run.
`N_YEARS` is the physical duration of the generated catalogue.

In [ ]:
CHAIN_PATH = ROOT / 'Tutorials' / 'Output_files' / 'tutorial2_structured' / 'emcee.h5'
BURN_IN = 300
N_YEARS = 10.
DATA_DIR = ROOT / 'datafiles'
MRD_PATH = DATA_DIR / 'MRD_outputs' / 'fiducial_A1_0_BNS.csv'
STRUCTURE_CSV = None      # Tutorial 2; for 2.1 use DATA_DIR / 'structure_constants/struct_results_100.csv'
THETA_V_MAX_DEG = 30.
SEED = 42
N_JOBS = 4               # Set to 1 for a serial run.
OUTPUT_DIR = ROOT / 'Output_files' / 'tutorial6_forecasts'

PROMPT = (
    fp.PromptInstrument('GRINTA_TED', (50., 300.), 1., 0.35, 'ph', 0.6),
    fp.PromptInstrument('Crystal_Eye', (50., 300.), 2., 0.455, 'ph', 0.3),
    fp.PromptInstrument('THESEUS_XGIS', (30., 150.), 1., 3e-8, 'erg', 0.16 * 0.65, 15., 0.9),
)
MAX_T90_S = None          # Set to 2.0 if you want T90 < 2 s; otherwise apply flux cuts only.

NETWORKS = {
    'ET_triangle': ['ET'],
    'ET_2L': ['ETS_15', 'ETN_15'],
    'Asharp': ['LLO_sharp', 'LHO_sharp', 'LIO_sharp'],
    'ET_CE': ['ET', 'CE40'],
}
WAVEFORM = 'IMRPhenomD_NRTidalv2'    # Or 'TaylorF2'.
GW_SNR_THRESHOLD = 8.
LOCALIZATION_CUTS_DEG2 = (1., 10., 100., 1000.)

AFTERGLOW_IMAGERS = fp.DEFAULT_AFTERGLOW
RESPONSE_DELAYS_S = (0., 100., 300., 3600.)
GW_FOLLOWUP_AREA90_MAX_DEG2 = 100.
AFTERGLOW_THETA_CORE_DEG = 5.

## 2. Load the chain and initialize the fitted population

As in Tutorial 2.1, use the retained sample with the highest log probability.
All seven parameters, including $f_j$, come from that sample.

In [ ]:
backend = emcee.backends.HDFBackend(str(CHAIN_PATH), read_only=True)
chain = backend.get_chain(discard=BURN_IN, flat=True)
log_probability = backend.get_log_prob(discard=BURN_IN, flat=True)
theta = chain[np.argmax(log_probability)]
parameter_names = ['k', 'L_L0', 'L_mu_E', 'sigma_E', 'L_mu_tau', 'sigma_tau', 'f_j']
fitted_parameters = pd.Series(theta, index=parameter_names, name='fitted value')
display(fitted_parameters.to_frame())

simulation, interps, _ = initialize_simulation(
    DATA_DIR, MRD_PATH, params={'theta_v_max': THETA_V_MAX_DEG}, size_test=10_000)
if STRUCTURE_CSV is not None:
    simulation = load_custom_structure_constants(simulation, STRUCTURE_CSV)
simulation.rng = np.random.default_rng(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fitted_parameters.to_csv(OUTPUT_DIR / 'fitted_parameters.csv')

## 3. Generate the full catalogue and observables

Sample all jet sources over `N_YEARS` within the chosen viewing-angle range,
using the fitted population and MAGGPY's spectral and temporal model. The catalogue
includes redshift, viewing angle, peak energy, flux, fluence, duration and intrinsic
isotropic energy. Generation runs in small batches to limit memory use.

Instrument coverage is applied in the next step, so every annual rate is simply
the selected number of events divided by `N_YEARS`.

In [ ]:
catalogue = fp.make_full_catalogue(theta, simulation, interps, years=N_YEARS)
catalogue.to_csv(OUTPUT_DIR / 'full_catalogue.csv', index=False)
display(catalogue.head())

## 4. Prompt detections

Compare the maximum window-averaged flux with each threshold: TED uses 1 s in
50–300 keV, Crystal Eye 2 s in 50–300 keV, and XGIS 1 s in 30–150 keV.
GRINTA uses GBM-like coverage (0.6); Crystal Eye uses half that value.
XGIS uses an imaging sky fraction of 0.16 and an editable duty assumption of 0.65.
Its flux is integrated as energy flux; the other two use photon flux.

In [ ]:
display(pd.DataFrame([asdict(instrument) for instrument in PROMPT]))
events = fp.select_prompt(catalogue, instruments=PROMPT, seed=SEED, max_t90_s=MAX_T90_S)
prompt_table = fp.prompt_summary(events, N_YEARS, instruments=PROMPT)
display(prompt_table)
prompt_table.to_csv(OUTPUT_DIR / 'prompt_rates.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bins = {'z': np.linspace(0, events.z.max(), 25),
        'theta_v_deg': np.linspace(0, THETA_V_MAX_DEG, 25)}
for instrument in PROMPT:
    selected = events[f'prompt_{instrument.name}']
    for ax, column in zip(axes, ('z', 'theta_v_deg')):
        counts, edges = np.histogram(events.loc[selected, column], bins=bins[column])
        ax.stairs(counts / N_YEARS, edges, label=instrument.name)
axes[0].set(xlabel='Redshift', ylabel='Detections / year / bin')
axes[1].set(xlabel='Viewing angle [deg]', ylabel='Detections / year / bin')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'prompt_distributions.pdf')
plt.show()

## 5. Joint GW detections and localizations

Use the catalogue's redshift and viewing angle for each BNS. Add the tutorial mass
distribution ($1.33\pm0.09\,M_\odot$), sky position, polarization and coalescence time.
GWFish computes the SNRs and 90% Fisher sky areas. The comparison assumes all GW
detectors are operational. Sources without prompt triggers are retained for the
GW-triggered afterglow analysis.

The joint table reports annual rates and GW localization counts below each area cut.
XGIS imaging benchmarks are ≤7 arcmin for 50% and ≤15 arcmin for 90% of triggered
sGRBs; these are mission requirements rather than fitted per-event positions.

In [ ]:
binaries = fp.binary_parameters(events, seed=SEED)
detector_names = sorted({detector for network in NETWORKS.values() for detector in network})
gw_config = fp.resolve_gw_config(
    ROOT / 'Tutorials' / 'configs' / 'coba.yaml', ROOT / 'Tutorials' / 'psds',
    OUTPUT_DIR / 'detectors.yaml', detector_names)
gw_results = fp.run_gw(binaries, NETWORKS, gw_config, waveform=WAVEFORM,
                       threshold=GW_SNR_THRESHOLD, n_jobs=N_JOBS)
joint_table = fp.joint_summary(events, gw_results, N_YEARS, instruments=PROMPT,
                                localization_cuts=LOCALIZATION_CUTS_DEG2)
display(joint_table)
joint_table.to_csv(OUTPUT_DIR / 'joint_prompt_rates.csv', index=False)
pd.concat([events, binaries.add_prefix('bns_'), gw_results], axis=1).to_csv(
    OUTPUT_DIR / 'events_prompt_gw.csv', index=False)

ax = joint_table.pivot(index='network', columns='instrument', values='rate_per_yr').plot.bar(
    figsize=(10, 4), rot=0, ylabel='Prompt + GW detections / year')
ax.legend(title='Prompt instrument', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'joint_prompt_rates.pdf')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for instrument in PROMPT:
    for network in NETWORKS:
        joint = events[f'prompt_{instrument.name}'] & gw_results[f'gw_{network}']
        area = gw_results.loc[joint, f'area90_{network}_deg2'].dropna().sort_values()
        if len(area):
            ax.step(area, np.arange(1, len(area) + 1) / N_YEARS, where='post',
                    label=f'{instrument.name} + {network}')
ax.set(xscale='log', xlabel='90% GW sky area [deg²]', ylabel='Joint detections / year below area')
if ax.get_legend_handles_labels()[0]:
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'joint_localization_cumulative.pdf')
plt.show()

## 6. Afterglows and additional counterparts

VegasAfterglow uses the same source's $E_{\rm iso}$, redshift and viewing angle,
with the supplied 5° top-hat afterglow setup. The density, $\epsilon_B$ and Lorentz
factor are drawn once per source and reused across the comparisons.

HXI uses 5–30 keV and $1.2\times10^{-3}\sqrt{10^4/\Delta t}$ ph cm$^{-2}$ s$^{-1}$.
SXI uses the [Yellow Book](https://sci.esa.int/documents/34375/36249/Theseus_YB_final.pdf)
sensitivities: $10^{-10}$ erg cm$^{-2}$ s$^{-1}$ at 100 s and
$1.8\times10^{-11}$ at 1,500 s, in 0.3–5 keV. Each exposure starts after its
alert and response delay. The calculation finds the best exposure during the light curve.

`followup_fraction=1` assumes that follow-up is arranged; edit the imager values
to include availability. GW follow-up requires the selected localization cut.
The rate table reports prompt + GW events with afterglows, GW-triggered afterglows,
and the **additional events without a prompt detection** in that instrument.

In [ ]:
display(pd.DataFrame([asdict(imager) for imager in AFTERGLOW_IMAGERS]))
microphysics = fp.afterglow_parameters(events, seed=SEED)
afterglow_events = fp.run_afterglow(
    events, gw_results, microphysics, instruments=AFTERGLOW_IMAGERS,
    prompt_instruments=PROMPT, delays=RESPONSE_DELAYS_S,
    localization_cut=GW_FOLLOWUP_AREA90_MAX_DEG2,
    theta_core_deg=AFTERGLOW_THETA_CORE_DEG, n_jobs=N_JOBS)
afterglow_table = fp.afterglow_summary(
    events, gw_results, afterglow_events, N_YEARS,
    instruments=AFTERGLOW_IMAGERS, prompt_instruments=PROMPT,
    delays=RESPONSE_DELAYS_S, localization_cut=GW_FOLLOWUP_AREA90_MAX_DEG2, seed=SEED)
display(afterglow_table)
afterglow_table.to_csv(OUTPUT_DIR / 'afterglow_rates.csv', index=False)
afterglow_events.to_csv(OUTPUT_DIR / 'afterglow_events.csv', index=False)
microphysics.to_csv(OUTPUT_DIR / 'afterglow_parameters.csv', index=False)

fig, axes = plt.subplots(1, len(AFTERGLOW_IMAGERS), figsize=(11, 4), squeeze=False)
for ax, imager in zip(axes[0], AFTERGLOW_IMAGERS):
    rows = afterglow_table[afterglow_table.imager == imager.name]
    for (instrument, network), group in rows.groupby(['prompt_instrument', 'network']):
        group = group.sort_values('delay_s')
        ax.plot(group.delay_s, group.extra_without_prompt_per_yr, 'o-', label=f'{instrument} + {network}')
    ax.set(xlabel='Response delay [s]', ylabel='Additional detections / year', title=imager.name)
    ax.set_xscale('symlog', linthresh=10)
    ax.legend(fontsize=6, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'extra_afterglow_rates.pdf')
plt.show()

## 7. Numbers for the chapter

Combine the joint rates, GW localizations and afterglow scenarios in one table.
`unique_prompt_or_added_afterglow` counts each counterpart once: prompt + GW
events plus the exclusive additions from GW-triggered afterglows.

In [ ]:
chapter_numbers = joint_table.rename(columns={
    'instrument': 'prompt_instrument', 'count': 'prompt_joint_count',
    'rate_per_yr': 'prompt_joint_per_yr'})
chapter_numbers = chapter_numbers.merge(afterglow_table.drop(columns='prompt_joint_count'),
                                         on=['prompt_instrument', 'network'])
chapter_numbers.to_csv(OUTPUT_DIR / 'chapter_numbers.csv', index=False)
display(chapter_numbers)
print(f'Saved catalogues, rate tables and figures in {OUTPUT_DIR}')